# 39. SegFormer 간단 실습

이 노트북은 `38_SegFormer_전체_Forward_흐름.ipynb` 다음 단계로, synthetic image-mask pair를 사용해 segmentation 입력, label, prediction, metric 계산 흐름을 실습합니다.

실제 SegFormer 학습은 PyTorch와 데이터셋이 필요하지만, 여기서는 외부 데이터 없이 semantic segmentation 실험의 기본 감각을 익히는 데 집중합니다.

이번 노트북의 목표는 다음과 같습니다.

- image-mask pair의 형태를 확인합니다.
- class map과 color map을 시각화합니다.
- 간단한 pixel classifier로 segmentation prediction을 만들어 봅니다.
- pixel accuracy와 mean IoU를 계산합니다.

In [ ]:
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(3)

## 39-1. Synthetic image-mask pair 만들기

배경, 원형 객체, 사각형 객체로 이루어진 작은 segmentation 데이터 하나를 만듭니다. class id는 다음과 같습니다.

- 0: background
- 1: circle
- 2: rectangle

In [ ]:
h, w = 128, 128
image = np.zeros((h, w, 3), dtype=np.float32)
mask = np.zeros((h, w), dtype=np.int64)

image[:, :, :] = [0.72, 0.78, 0.84]
yy, xx = np.ogrid[:h, :w]
circle = (xx - 42) ** 2 + (yy - 62) ** 2 < 25 ** 2
rect = (xx >= 70) & (xx <= 112) & (yy >= 38) & (yy <= 94)

image[circle] = [0.90, 0.28, 0.22]
image[rect] = [0.20, 0.48, 0.90]
mask[circle] = 1
mask[rect] = 2

noise = np.random.normal(scale=0.035, size=image.shape)
image = np.clip(image + noise, 0, 1)

cmap = ListedColormap(['#94a3b8', '#ef4444', '#3b82f6'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image)
axes[0].set_title('image')
axes[1].imshow(mask, cmap=cmap, vmin=0, vmax=2)
axes[1].set_title('label mask')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 39-2. 간단한 pixel classifier

실제 SegFormer는 image context와 multi-scale feature를 사용합니다. 여기서는 RGB 값과 위치 정보를 feature로 사용한 간단한 classifier로 segmentation metric 계산 흐름을 연습합니다.

In [ ]:
y_grid, x_grid = np.mgrid[0:h, 0:w]
features = np.concatenate([
    image.reshape(-1, 3),
    (x_grid / w).reshape(-1, 1),
    (y_grid / h).reshape(-1, 1),
    np.ones((h * w, 1)),
], axis=1)
target = mask.reshape(-1)

num_classes = 3
one_hot = np.eye(num_classes)[target]

# ridge regression 형태의 매우 단순한 linear classifier
reg = 1e-3
weights = np.linalg.solve(features.T @ features + reg * np.eye(features.shape[1]), features.T @ one_hot)
logits = features @ weights
pred = logits.argmax(axis=1).reshape(h, w)

print('features:', features.shape)
print('weights:', weights.shape)
print('prediction:', pred.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(image)
axes[0].set_title('image')
axes[1].imshow(mask, cmap=cmap, vmin=0, vmax=2)
axes[1].set_title('ground truth')
axes[2].imshow(pred, cmap=cmap, vmin=0, vmax=2)
axes[2].set_title('prediction')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 39-3. Pixel accuracy와 mean IoU

Semantic segmentation에서는 픽셀 단위 정확도뿐 아니라 class별 IoU를 자주 봅니다.

```text
IoU = intersection / union
mean IoU = class별 IoU 평균
```

In [ ]:
def pixel_accuracy(pred, target):
    return (pred == target).mean()

def class_iou(pred, target, cls):
    pred_c = pred == cls
    target_c = target == cls
    inter = np.logical_and(pred_c, target_c).sum()
    union = np.logical_or(pred_c, target_c).sum()
    return inter / union if union > 0 else np.nan

ious = [class_iou(pred, mask, cls) for cls in range(num_classes)]
print('pixel accuracy:', round(pixel_accuracy(pred, mask), 4))
for cls, iou in enumerate(ious):
    print(f'class {cls} IoU:', round(iou, 4))
print('mean IoU:', round(np.nanmean(ious), 4))

## 39-4. 실제 SegFormer 학습으로 바꿀 때

실제 코드에서는 위 linear classifier 부분이 SegFormer model forward로 바뀝니다.

```python
outputs = model(pixel_values=image_tensor)
logits = outputs.logits
loss = criterion(logits, mask_tensor)
```

하지만 입력 image, label mask, logits, prediction, IoU 계산 흐름은 동일합니다.

## 정리

- Semantic segmentation 데이터는 image와 같은 공간 크기를 가진 class mask로 구성됩니다.
- 모델 출력 logits는 class dimension을 가지며, argmax로 prediction map을 얻습니다.
- pixel accuracy는 전체 픽셀 중 맞춘 비율이고, mean IoU는 class별 영역 겹침을 평균한 값입니다.
- 다음 노트북 `40_SegFormer_논문_구조_정리와_개발자_관점.ipynb`에서는 SegFormer를 개발자 관점에서 정리합니다.